# MedAgentBench GRPO Training with Unsloth + TRL

Fine-tune **Qwen3-1.7B** on the MedAgentBench clinical decision-making benchmark using:
- **Unsloth** for 2-4x faster LoRA fine-tuning
- **TRL GRPOTrainer** with environment `tool-use` rollouts
- **Mock FHIR** backend — no live server, runs fully offline
- **Named tool calls** matching benchmark evaluation format

**Runtime:** T4 minimum, A100/H100 recommended

## 1. Install Dependencies & Clone Repo

In [ ]:
# Install training stack
!pip install -q unsloth trl datasets huggingface_hub

# Clone repo (contains medagentbench_env + benchmark data)
import os
if not os.path.exists('./repo'):
    !git clone https://github.com/ananya173147/clinKriya.git ./repo
else:
    !git -C ./repo pull

!nvidia-smi | head -20

## 2. Load Model with Unsloth (4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "Qwen/Qwen3-1.7B"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,  # must match bf16=True in GRPOConfig
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

print(f"Model: {MODEL_NAME}")
model.print_trainable_parameters()

## 3. Load Environment & Data

In [ ]:
import importlib.util
from pathlib import Path

REPO = Path("./repo")

# Load train.py directly — avoids installing openenv-core
spec = importlib.util.spec_from_file_location(
    "train", REPO / "medagentbench_env" / "train.py"
)
train_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_mod)

MedAgentTrainEnv = train_mod.MedAgentTrainEnv
reward_func      = train_mod.reward_func
build_dataset    = train_mod.build_dataset

DATA_DIR = REPO / "medagentbench_env" / "data"

# Point module at cloned data
train_mod._DATA_DIR            = DATA_DIR
train_mod._CACHE_PATH          = DATA_DIR / "fhir_cache.json"
train_mod._SYSTEM_PROMPT_PATH  = DATA_DIR / "new_system.txt"

# Pre-load shared resources
train_mod._get_mock_fhir()
tasks = train_mod._get_tasks()
print(f"FHIR cache loaded | {len(tasks)} tasks")
print(f"System prompt (first 80 chars): {train_mod._get_system_prompt()[:80]}...")

In [ ]:
# Sanity check — run one episode end-to-end
env = MedAgentTrainEnv()
instruction = env.reset()
task = env._task
print(f"Task : {task['id']}  ({task.get('_benchmark_type')})")
print(f"Instr: {instruction[:120]}...")

# Simulate a correct BP observation POST
resp = env.fhir_vitals_create(
    resourceType="Observation",
    category=[{"coding": [{"code": "vital-signs"}]}],
    code={"text": "BP"},
    effectiveDateTime="2023-11-13T10:15:00",
    status="final",
    valueString="118/77",
    subject={"reference": f"Patient/{task['eval_MRN']}"},
)
print(f"POST: {resp[:60]}")

result = env.finish([])
print(result)

## 4. Build Training Dataset

In [ ]:
dataset = build_dataset(DATA_DIR)
print(f"Dataset: {len(dataset)} tasks")
print(f"Roles  : {[m['role'] for m in dataset[0]['prompt']]}")
print(f"User   : {dataset[0]['prompt'][1]['content']}")

## 5. Train with GRPOTrainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "./medagent_grpo_output"

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_completion_length=2048,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_steps=10,
    logging_steps=1,
    log_completions=True,
    num_completions_to_print=2,
    bf16=True,
    save_steps=50,
    save_total_limit=2,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    train_dataset=dataset,
    environment_factory=MedAgentTrainEnv,
    processing_class=tokenizer,
    args=grpo_config,
)

print("Starting GRPO training...")
trainer.train()

## 6. Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to {OUTPUT_DIR}")

# Merge LoRA into base weights (needed for full model push)
# merged = model.merge_and_unload()
# merged.save_pretrained(OUTPUT_DIR + "_merged")
# tokenizer.save_pretrained(OUTPUT_DIR + "_merged")

## 7. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login

HF_TOKEN   = "hf_xxx"                          # your HF write token
HF_REPO_ID = "YOUR_USERNAME/medagent-qwen3-1.7b"

login(token=HF_TOKEN)

model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

## 8. Quick Evaluation

Run the trained model on a sample of tasks. Parses Qwen3 `<tool_call>` format
and routes to environment methods.

In [ ]:
import json, re

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (REPO / "medagentbench_env" / "data" / "new_system.txt").read_text().strip()


def parse_tool_call(text: str):
    """Parse Qwen3 <tool_call>{...}</tool_call> format."""
    m = re.search(r'<tool_call>\s*({.*?})\s*</tool_call>', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # Fallback: bare {"name": ..., "arguments": ...}
    m = re.search(r'\{\s*"name"\s*:\s*"(\w+)"\s*,\s*"arguments"\s*:\s*(\{.*?\})\s*\}',
                  text, re.DOTALL)
    if m:
        try:
            return {"name": m.group(1), "arguments": json.loads(m.group(2))}
        except json.JSONDecodeError:
            pass
    return None


def run_episode(env, max_steps=8):
    """Run one episode with the trained model, return reward."""
    instruction = env.reset()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": instruction},
    ]

    for step in range(max_steps):
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
        reply = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

        tc = parse_tool_call(reply)
        if tc is None:
            env.finish([])
            break

        tool_name = tc.get("name", "")
        tool_args = tc.get("arguments", {})
        method = getattr(env, tool_name, None)

        if method is None or tool_name.startswith("_"):
            env.finish([])
            break

        try:
            tool_result = method(**tool_args)
        except Exception as e:
            tool_result = f"Tool error: {e}"

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "tool", "content": tool_result,
                         "tool_call_id": tool_name})

        if env.done:
            break

    return env.reward


train_mod._TASK_INDEX = 0
NUM_EVAL = 10
rewards = []

for i in range(NUM_EVAL):
    env = MedAgentTrainEnv()
    r = run_episode(env)
    rewards.append(r)
    print(f"  {env._task['id']}: reward={r:.3f}")

avg = sum(rewards) / len(rewards)
print(f"\nPost-training avg reward ({NUM_EVAL} tasks): {avg:.4f}")
print(f"Baseline (Qwen3-8B OpenRouter):             0.2748")

## 9. Load from HuggingFace (clone)

Re-load the pushed model on any machine — no repo clone needed.

In [ ]:
# from unsloth import FastLanguageModel
#
# HF_REPO_ID = "YOUR_USERNAME/medagent-qwen3-1.7b"
#
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=HF_REPO_ID,
#     max_seq_length=4096,
#     load_in_4bit=True,
#     dtype=torch.bfloat16,
# )
# FastLanguageModel.for_inference(model)
# print(f"Loaded {HF_REPO_ID} from HuggingFace Hub")